# Performance Metrics

Performance metrics summarize how a portfolio behaved after considering return, risk, downside, benchmark comparison, and consistency.

Abbreviations used in this notebook:

- **CAGR**: Compound Annual Growth Rate.
- **IR**: Information Ratio.
- **TE**: Tracking Error.
- **VaR**: Value at Risk.
- **CVaR**: Conditional Value at Risk.
- **MDD**: Maximum Drawdown.

## 1. Intuition

No single metric tells the full story. CAGR describes growth, volatility describes variability, Sharpe ratio describes excess return per unit of volatility, drawdown describes pain, and tracking error describes benchmark-relative risk.

## 2. Mathematics

Sharpe ratio:

$$
Sharpe = \frac{R_p - R_f}{\sigma_p}
$$

Tracking error:

$$
TE = SD(R_p - R_b) \times \sqrt{252}
$$

Information ratio:

$$
IR = \frac{R_p - R_b}{TE}
$$

Maximum drawdown:

$$
MDD = \min_t \left(\frac{Wealth_t}{Peak_t} - 1\right)
$$

Conditional VaR:

$$
CVaR = -E[R_t | R_t \le Percentile(R, 5\%)]
$$

Where:
- `R_p` = portfolio return.
- `R_f` = risk-free return.
- `sigma_p` = portfolio volatility.
- `TE` = tracking error, the volatility of active returns.
- `R_b` = benchmark return.
- `IR` = information ratio.
- `MDD` = maximum drawdown.
- `CVaR` = conditional value at risk.


## 3. Implementation

We calculate performance metrics for an equal-weight benchmark and a simple diversified portfolio.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "03_portfolio_management" / "portfolio_utils.py"
spec = importlib.util.spec_from_file_location("portfolio_utils", helper_path)
portfolio_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(portfolio_utils)

plt.style.use("seaborn-v0_8-whitegrid")

returns = portfolio_utils.generate_synthetic_returns(periods=504, seed=321)
prices = portfolio_utils.returns_to_prices(returns)
assets = returns.columns.tolist()

benchmark_weights = np.repeat(1 / len(assets), len(assets))
portfolio_weights = np.array([0.32, 0.23, 0.25, 0.12, 0.08])

benchmark = portfolio_utils.portfolio_series(returns, benchmark_weights)
portfolio = portfolio_utils.portfolio_series(returns, portfolio_weights)

comparison = pd.DataFrame({"portfolio": portfolio, "benchmark": benchmark})
comparison.head()

In [ ]:
def performance_metrics(series, benchmark_series=None, risk_free_rate=0.015):
    ann_return = portfolio_utils.annualized_return(series)
    ann_vol = portfolio_utils.annualized_volatility(series)
    dd = portfolio_utils.drawdown(series)["drawdown"]
    metrics = {
        "cagr": ann_return,
        "annualized_volatility": ann_vol,
        "sharpe": portfolio_utils.sharpe_ratio(ann_return, ann_vol, risk_free_rate),
        "daily_var_5pct": -series.quantile(0.05),
        "daily_cvar_5pct": -series[series <= series.quantile(0.05)].mean(),
        "max_drawdown": dd.min(),
        "positive_day_ratio": (series > 0).mean(),
    }
    if benchmark_series is not None:
        active = series - benchmark_series
        tracking_error = active.std() * np.sqrt(252)
        active_return = portfolio_utils.annualized_return(series) - portfolio_utils.annualized_return(benchmark_series)
        metrics["tracking_error"] = tracking_error
        metrics["information_ratio"] = active_return / tracking_error if tracking_error else np.nan
    return pd.Series(metrics)

metrics = pd.DataFrame({
    "portfolio": performance_metrics(portfolio, benchmark),
    "benchmark": performance_metrics(benchmark),
})

metrics.round(4)

## 4. Visualization

Performance analysis should include wealth, drawdown, and rolling risk-adjusted performance.

In [ ]:
wealth = (1 + comparison).cumprod()
rolling_sharpe = (comparison.rolling(63).mean() * 252 - 0.015) / (comparison.rolling(63).std() * np.sqrt(252))

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
wealth.plot(ax=axes[0], color=["#2f6f8f", "#9a6b2f"])
axes[0].set_title("Wealth Curves")
axes[0].set_ylabel("Growth of 1")

portfolio_utils.drawdown(portfolio)["drawdown"].plot(ax=axes[1], color="#2f6f8f", label="Portfolio")
portfolio_utils.drawdown(benchmark)["drawdown"].plot(ax=axes[1], color="#9a6b2f", label="Benchmark")
axes[1].set_title("Drawdowns")
axes[1].set_ylabel("Drawdown")
axes[1].yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
axes[1].legend()

rolling_sharpe.plot(ax=axes[2], color=["#2f6f8f", "#9a6b2f"])
axes[2].set_title("63-Day Rolling Sharpe Ratio")
axes[2].set_ylabel("Sharpe")
axes[2].set_xlabel("Date")

plt.tight_layout()
plt.show()

In [ ]:
metrics.loc[["cagr", "annualized_volatility", "sharpe", "max_drawdown"]].T.plot(kind="bar", figsize=(10, 4))
plt.title("Metric Comparison")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Application

Performance metrics help decide whether a portfolio is doing what it was designed to do. A defensive allocation should usually show lower volatility and drawdown, even if it gives up some upside.

In [ ]:
active_return = portfolio_utils.annualized_return(portfolio) - portfolio_utils.annualized_return(benchmark)
tracking_error = (portfolio - benchmark).std() * np.sqrt(252)

application_summary = pd.Series({
    "active_return": active_return,
    "tracking_error": tracking_error,
    "information_ratio": active_return / tracking_error,
    "portfolio_weight_bonds": portfolio_weights[assets.index("Bonds")],
    "portfolio_weight_equities": portfolio_weights[assets.index("Global Equity")] + portfolio_weights[assets.index("Swiss Equity")],
})

application_summary.to_frame("value")

## 6. Reflection

- CAGR without risk is incomplete.
- Sharpe ratio is useful but assumes volatility is an adequate risk proxy.
- Drawdown captures path risk and investor discomfort.
- Benchmark-relative metrics matter when the portfolio has a mandate.
- Metrics should be interpreted together, not ranked blindly.

Questions to answer after running the notebook:

1. Did the portfolio improve risk-adjusted performance?
2. Was lower drawdown worth any return trade-off?
3. Which metric would matter most to a conservative investor?
4. What metric would you add for a strategy with monthly withdrawals?